# 01 — Разведочный анализ данных (EDA)

**Проект:** прогнозирование оттока клиентов телеком-оператора.

**Цель ноутбука:** изучить синтетический датасет — распределения признаков,
пропуски, несбалансированность классов и связь признаков с оттоком.

**Данные:** синтетический датасет (`src/data/generate.py`), 7000 клиентов.
Запускать ноутбук из каталога `project/`.

In [ ]:
import sys
# импорт кода проекта (ноутбук запускается из каталога project/)
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generate import generate_dataset
from src.data.loader import NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_COLUMN

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Загрузка данных

In [ ]:
df = generate_dataset(n_customers=7000, random_state=42, target_churn_rate=0.26)
print("Размер датасета:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Несбалансированность классов

Целевая переменная `churn` — уйдёт ли клиент. Проверим баланс классов.

In [ ]:
churn_rate = df[TARGET_COLUMN].mean()
print(f"Доля оттока: {churn_rate:.3f}")
print(df[TARGET_COLUMN].value_counts())

ax = df[TARGET_COLUMN].value_counts().sort_index().plot(
    kind="bar", color=["#22c55e", "#ef4444"], figsize=(5, 4),
)
ax.set_xticklabels(["Остался (0)", "Ушёл (1)"], rotation=0)
ax.set_title("Распределение оттока")
plt.show()

**Вывод:** классы несбалансированы — отток составляет около 26%.
Это обосновывает выбор метрик ROC-AUC / PR-AUC и использование
`class_weight="balanced"` для линейной модели и леса.

## 3. Анализ пропусков

In [ ]:
missing = df.isna().sum()
print("Пропуски по колонкам:")
print(missing[missing > 0])

# где именно пропуски в total_charges?
na_rows = df[df["total_charges"].isna()]
print(f"\nИз {len(na_rows)} строк с пропуском total_charges "
      f"у {(na_rows['tenure'] == 0).sum()} клиентов tenure == 0")

**Вывод:** пропуски есть только в `total_charges`, преимущественно у клиентов
с нулевым сроком обслуживания (новые клиенты ещё ничего не заплатили).
Решается импьютацией медианой на этапе препроцессинга.

## 4. Распределения числовых признаков

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), NUMERIC_FEATURES):
    df[col].hist(bins=30, ax=ax, color="#2563eb")
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

## 5. Связь признаков с оттоком

In [ ]:
# доля оттока по категориальным признакам
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.ravel(), CATEGORICAL_FEATURES):
    rates = df.groupby(col)[TARGET_COLUMN].mean().sort_values()
    rates.plot(kind="barh", ax=ax, color="#ef4444")
    ax.set_title(f"Доля оттока по '{col}'")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# числовые признаки: распределение по классам
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["tenure", "monthly_charges", "num_support_calls"]):
    sns.boxplot(data=df, x=TARGET_COLUMN, y=col, ax=ax)
    ax.set_title(col)
    ax.set_xticklabels(["Остался", "Ушёл"])
plt.tight_layout()
plt.show()

In [ ]:
# корреляция числовых признаков с оттоком
corr = df[NUMERIC_FEATURES + [TARGET_COLUMN]].corr()[TARGET_COLUMN].sort_values()
print("Корреляция числовых признаков с churn:")
print(corr)

## 6. Выводы по EDA

1. **Дисбаланс классов** — отток ~26%; нужны устойчивые к дисбалансу метрики.
2. **Пропуски** — только в `total_charges`, в основном у новых клиентов;
   решаются импьютацией.
3. **Сильные предикторы оттока:**
   - помесячный контракт (`Month-to-month`) — резко повышенный отток;
   - короткий срок обслуживания (`tenure`);
   - оптоволоконный интернет и высокие ежемесячные траты;
   - большое число обращений в поддержку.
4. **Защитные факторы** — долгие контракты, длительный срок обслуживания.

Эти наблюдения подтверждаются значимостью признаков финальной модели
(см. ноутбук `03_models_tuning.ipynb`).